# Bronze Ingestion: Nest HVAC Runtime

This notebook reads Nest thermostat HVAC runtime data from a landing volume (CSV-quoted JSONL files),
unescape the CSV quoting, parses the JSON into structured columns, and writes the result to a bronze
Delta table with provenance metadata.

**Source**: CSV-quoted JSONL files in a UC volume
**Target**: Bronze Delta table with `_source_file`, `_device_id_from_filename`, and `_ingest_timestamp` metadata columns

**Parameters**:
- `catalog`: Unity Catalog catalog name
- `schema`: Unity Catalog schema name
- `bronze_table`: Name of the bronze Delta table to write
- `volume_path`: Path to the landing volume containing JSONL files

In [0]:
# Widget parameters — set by the DAB job or manually when running interactively
dbutils.widgets.text("catalog", "workspace", "Unity Catalog catalog")
dbutils.widgets.text("schema", "barronfuentes", "Unity Catalog schema")
dbutils.widgets.text("bronze_table", "bronze_nest_hvac_runtime", "Bronze table name")
dbutils.widgets.text("volume_path", "/Volumes/workspace/barronfuentes/nest_thermostat_takeout", "Path to landing volume with JSONL files")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
bronze_table = dbutils.widgets.get("bronze_table")
volume_path = dbutils.widgets.get("volume_path")

full_table_name = f"{catalog}.{schema}.{bronze_table}"
print(f"Target table: {full_table_name}")
print(f"Source volume: {volume_path}")

In [0]:
from pyspark.sql.functions import col, regexp_replace, from_json, schema_of_json, current_timestamp, regexp_extract

# The JSONL files from Nest are CSV-quoted: each line is wrapped in double quotes
# with inner quotes doubled (e.g. "{""key"":""value""}"").
# We read as text first to preserve the raw lines, then unescape manually.
df_raw = spark.read.text(f"{volume_path}/*.jsonl")

print(f"Raw lines read: {df_raw.count()}")
df_raw.select("value").limit(2).show(truncate=200)

In [0]:
# Unescape CSV quoting in two steps:
# 1. Strip the outer wrapping quotes (first and last character of each line)
# 2. Replace doubled quotes ("") with single quotes (")
df_json = df_raw.select(
    col("_metadata.file_path").alias("_source_file"),
    regexp_replace(
        regexp_replace("value", '^"', ''), '"$', ''
    ).alias("json_str_1")
)

df_json = df_json.select(
    "_source_file",
    regexp_replace("json_str_1", '""', '"').alias("json_str")
)

# Verify the unescaping produced valid JSON
print("Sample unescaped JSON:")
df_json.select("json_str").limit(1).show(truncate=500)

In [0]:
# Infer the JSON schema from a single sample record, then parse all rows
sample_json = df_json.select("json_str").limit(1).collect()[0][0]
inferred_schema = schema_of_json(sample_json)
print(f"Inferred schema: {inferred_schema}")

df_parsed = df_json.select(
    "_source_file",
    from_json("json_str", inferred_schema).alias("data")
)

# Flatten the struct into individual columns
df_flat = df_parsed.select("_source_file", "data.*")

df_flat.printSchema()
print(f"Parsed rows: {df_flat.count()}")

In [0]:
# Add bronze-layer metadata columns:
# - _device_id_from_filename: device ID extracted from the source filename (provenance cross-check)
# - _ingest_timestamp: when this row was loaded into the bronze table
df_final = df_flat \
    .withColumn("_device_id_from_filename", regexp_extract("_source_file", r"/(\w+)_\d{4}_\d{2}_HvacRuntime\.jsonl", 1)) \
    .withColumn("_ingest_timestamp", current_timestamp())

# Write to the bronze Delta table (overwrite mode for full reload)
df_final.write.mode("overwrite").saveAsTable(full_table_name)

print(f"Bronze table written: {full_table_name}")
print(f"Total rows: {df_final.count()}")
print(f"Total columns: {len(df_final.columns)}")

In [0]:
# Verify the bronze table: row count per device and date range
spark.sql(f"""
SELECT device_id,
       COUNT(*) as records,
       MIN(interval_start) as earliest,
       MAX(interval_end) as latest
FROM {full_table_name}
GROUP BY device_id
ORDER BY device_id
""").display()